# Term Deposit Subscription Prediction

This notebook accompanies the executive report and implements the full data science workflow for the term deposit subscription prediction problem:

1. Executive Summary
2. Business Context and Problem Statement
3. Scope and Methodology
4. Exploratory Data Analysis (EDA)
5. Data Preprocessing
6. Modeling & Evaluation (Logistic Regression, Random Forest, Gradient Boosting, XGBoost)
7. Feature Importance and Interpretation
8. Conclusions and Next Steps

The goal is to build predictive models to estimate the probability that a client will subscribe to a term deposit during a direct marketing campaign, and to extract insights that can help the bank optimize its outbound calling strategy.

# **1. Executive Summary**

The purpose of this project is to develop a predictive model capable of estimating whether a bank client will subscribe to a term deposit during a direct marketing campaign. The dataset provided contains detailed demographic, behavioral, and campaign-related features collected by a Portuguese banking institution during multiple outbound phone marketing efforts. The model aims to support the marketing and operations teams by enabling intelligent prioritization of clients with higher likelihood of conversion, thereby increasing efficiency, reducing operational costs, and improving overall campaign performance.

A complete end-to-end data science workflow was implemented, including data exploration, preprocessing, feature engineering, model development, and evaluation. Four supervised machine learning models were compared: Logistic Regression (baseline), Random Forest, Gradient Boosting, and XGBoost. Evaluation metrics appropriate for imbalanced classification—such as ROC-AUC, precision, recall, and F1-score—were used to select the best performing model and understand trade-offs between false positives and false negatives.

Key findings show that the dataset exhibits strong class imbalance, with only about 11% of clients subscribing to the product. The variable *duration* is highly predictive but unsuitable for real-world pre-call prediction due to target leakage. Other features such as previous campaign outcome, economic indicators, age, and contact patterns play an important role in predicting subscription propensity. Tree-based ensemble models, especially Gradient Boosting and XGBoost, demonstrated the strongest predictive performance.

Based on the insights obtained, the bank can significantly improve its calling strategy by ranking clients according to predicted subscription probability, reducing unnecessary or low-yield call attempts, and tailoring communication strategies to specific customer segments. A high-level cloud deployment architecture (Azure or AWS) is also proposed to integrate the model with existing CRM and dialing systems. Generative AI is not recommended as the primary modeling technique, but it can complement the workflow by generating personalized call scripts or summarizing client history for call center agents.

Next steps involve operationalizing the model, integrating it into production systems, monitoring performance, and setting up a periodic retraining pipeline.

# **2. Business Context and Problem Statement**

The Portuguese bank in this case study uses direct marketing campaigns via phone calls to promote term deposits. These campaigns typically involve multiple contact attempts per customer, which increases operational cost and may cause customer fatigue if not carefully targeted. In the current workflow, many clients are contacted with similar priority, regardless of their likelihood to accept the offer. This may lead to inefficiencies, wasted call center capacity, and lower conversion rates.

A term deposit is a secure, interest-bearing product where the client deposits a fixed amount of money for a predefined period. Marketing such products requires identifying customers who value stability, long-term planning, or favorable economic conditions. Since outbound phone campaigns require human resources and time, the bank would benefit from a systematic way to identify high-potential clients *before* initiating the call.

The core business question becomes:

**Can we predict which clients are more likely to subscribe to a term deposit so that the marketing team can prioritize their outreach and optimize campaign performance?**

Solving this question aligns with key business goals:

* Increasing conversion rate
* Reducing unnecessary calls
* Improving customer experience
* Optimizing call center operations
* Enhancing marketing ROI

The main objective of the data science solution is therefore to build a predictive model that estimates the probability of subscription for each customer using the available historical and contextual data. This probability can then be used to rank customers and guide the outbound calling strategy.

The scope of this analysis does not include staffing forecasting, call center scheduling, or optimization of non-phone channels. The focus remains strictly on designing and evaluating a robust predictive model capable of supporting data-driven decision making for phone-based marketing campaigns.


# 3. Scope and Methodology

We follow a standard data science workflow:
1. Exploratory Data Analysis (EDA)
2. Data Preprocessing
3. Predictive Modeling
4. Feature Interpretation
5. Business Recommendations
6. Cloud Deployment Architecture



## 4. Exploratory Data Analysis – Key Findings

In [ ]:
#Libraries imports and dataset loading

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_fscore_support,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


In [ ]:
# ====================================================
# Load dataset
# ====================================================

DATA_PATH = "bank-additional-full.csv"

df = pd.read_csv(DATA_PATH, sep=';')
df.head()


In [ ]:
# ====================================================
# Overview of columns and dtypes
# ====================================================
df.info()


### 4.1 Dataset Overview


In [ ]:
# ====================================================
# Basic dataset overview
# ====================================================

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nTarget variable distribution (absolute):")
print(df['y'].value_counts())

print("\nTarget variable distribution (percent):")
print(df['y'].value_counts(normalize=True) * 100)

### 4.2 Data Quality and 'Unknown' Values

In [ ]:
# ====================================================
# Check 'unknown' frequencies in key categorical features
# ====================================================

categorical_cols = [
    'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'month',
    'day_of_week', 'poutcome'
]

for col in categorical_cols:
    unknown_count = (df[col] == 'unknown').sum() if 'unknown' in df[col].unique() else 0
    total = len(df)
    if unknown_count > 0:
        print(f"{col}: {unknown_count} 'unknown' ({unknown_count/total*100:.2f}%)")
    else:
        print(f"{col}: no 'unknown' values")

In [ ]:
# ====================================================
# Unique values for each categorical column
# ====================================================

for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts())

### 4.3 Target Leakage and 'duration'

In [ ]:
# ====================================================
# Inspect relationship between duration and target
# ====================================================

print(df['duration'].describe())

plt.figure(figsize=(8,5))
sns.boxplot(x='y', y='duration', data=df)
plt.title("Call duration vs subscription (y)")
plt.show()

# Probability of "yes" by duration bucket
duration_bins = pd.cut(df['duration'], bins=[0, 60, 180, 600, 1800, np.inf],
                       labels=['&lt;=1 min', '1-3 min', '3-10 min', '10-30 min', '&gt;30 min'])
prob_by_bin = df.groupby(duration_bins)['y'].apply(lambda x: (x == 'yes').mean())
print(prob_by_bin)

### 4.4 Behavioral and Contextual Patterns

#### Numeric distributions and correlations

In [ ]:
# ====================================================
# Numeric features overview & correlation
# ====================================================

numeric_cols = [
    'age', 'duration', 'campaign', 'pdays', 'previous',
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
    'euribor3m', 'nr.employed'
]

df[numeric_cols].describe()

In [ ]:
# ====================================================
# Correlation matrix (including y encoded as 0/1)
# ====================================================

df_corr = df.copy()
df_corr['y_binary'] = (df_corr['y'] == 'yes').astype(int)

corr = df_corr[numeric_cols + ['y_binary']].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title("Correlation matrix (numeric features + y)")
plt.show()

#### Effect of poutcome and campaign

In [ ]:
# ====================================================
# Probability of 'yes' by poutcome and campaign
# ====================================================

prob_by_poutcome = df.groupby('poutcome')['y'].apply(lambda x: (x == 'yes').mean())
print("Subscription rate by poutcome:")
print(prob_by_poutcome)

# Binning campaign
campaign_bins = pd.cut(df['campaign'], bins=[0,1,2,4,10, np.inf],
                       labels=['1', '2', '3-4', '5-10', '&gt;10'])
prob_by_campaign = df.groupby(campaign_bins)['y'].apply(lambda x: (x == 'yes').mean())
print("\nSubscription rate by number of contacts in campaign:")
print(prob_by_campaign)

#### Client's Profile (age, job, education):


In [ ]:
# ====================================================
# Subscription rate by age bin, job, education
# ====================================================

df_age = df.copy()
age_bins = pd.cut(df_age['age'], bins=[17,25,35,50,65,100],
                  labels=['18-25','26-35','36-50','51-65','66+'])
df_age['age_bin'] = age_bins

print("Subscription rate by age_bin:")
print(df_age.groupby('age_bin')['y'].apply(lambda x: (x == 'yes').mean()))

print("\nTop 10 jobs by subscription rate:")
print(df.groupby('job')['y'].apply(lambda x: (x == 'yes').mean()).sort_values(ascending=False))

print("\nSubscription rate by education:")
print(df.groupby('education')['y'].apply(lambda x: (x == 'yes').mean()).sort_values(ascending=False))

## 5. Data Preprocessing

### 5.1 Variable Selection

In [ ]:
# ====================================================
# Define features and target variable
# ====================================================

# Target
y = (df['y'] == 'yes').astype(int)

# Features: all except 'y' and 'duration'
X = df.drop(columns=['y', 'duration'])

# Identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

### 5.2 Handling 'Unknown' Categories

In [ ]:
for col in categorical_features:
    if 'unknown' in X[col].unique():
        print(f"{col} has 'unknown' category (kept as is).")

### 5.3 Encoding categorical variables and scaling

In [ ]:
# ====================================================
# preprocessing pipeline (OneHot encoding + Scaling)
# ====================================================

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

### 5.5 Train/Test Split and Validation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

## 6. Modeling and Evaluation

In this section, we train and evaluate several classification models:

1. Logistic Regression (interpretable baseline)
2. Random Forest (bagging ensemble of trees)
3. Gradient Boosting (boosting ensemble)
4. XGBoost (advanced gradient boosting for tabular data)

We will:
- Define all models using `Pipeline` with the shared preprocessor
- Use a common evaluation function
- Compare models using ROC-AUC, precision, recall, and F1-score (positive class)
- Plot ROC curves for all models
- Show the confusion matrix for the best model
- Optionally compute cross-validated ROC-AUC on the training set

### 6.1 Logistic Regression (baseline)


In [ ]:
log_reg_clf = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs'))
    ]
)

log_reg_clf.fit(X_train, y_train)

y_pred_lr = log_reg_clf.predict(X_test)
y_proba_lr = log_reg_clf.predict_proba(X_test)[:, 1]

print("Logistic Regression - classification report:")
print(classification_report(y_test, y_pred_lr, digits=3))

print("Logistic Regression - ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

#### ROC curve for Logistic Regression

In [ ]:

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)

plt.figure(figsize=(6,6))
plt.plot(fpr_lr, tpr_lr, label='Logistic Regression')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.grid(True)
plt.show()


### Confusion Matrix





In [ ]:
# ====================================================
# Confusion matrix Logistic Regression
# ====================================================

cm_lr = confusion_matrix(y_test, y_pred_lr)

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [ ]:
from xgboost import XGBClassifier

# Class imbalance ratio for XGBoost scale_pos_weight

pos_rate = y_train.mean()
neg_rate = 1 - pos_rate
scale_pos_weight = neg_rate / pos_rate

print(f"Positive rate in training set: {pos_rate:.3f}")
print(f"Computed scale_pos_weight for XGBoost: {scale_pos_weight:.2f}")


In [ ]:
# Define all models in a dictionary

models = {}

# 1) Logistic Regression (interpretable baseline)
models["Logistic Regression"] = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            solver='lbfgs',
            n_jobs=-1
        ))
    ]
)

# 2) Random Forest (bagging ensemble of decision trees)
models["Random Forest"] = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced_subsample'
        ))
    ]
)

# 3) Gradient Boosting (boosting ensemble)
models["Gradient Boosting"] = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ]
)

# 4) XGBoost (advanced gradient boosting implementation for tabular data)
models["XGBoost"] = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
            scale_pos_weight=scale_pos_weight,
            tree_method="hist"
        ))
    ]
)


In [ ]:
# Helper function to train and evaluate a model

def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test):
    """
    Trains the given pipeline, evaluates it on the test set,
    and returns a dictionary with the main metrics.
    Also prints a human-readable summary.
    """
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # Core metrics
    roc = roc_auc_score(y_test, y_proba)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='binary', zero_division=0
    )

    print(f"\n================= {name} =================")
    print("Classification report:")
    print(classification_report(y_test, y_pred, digits=3))
    print(f"ROC-AUC        : {roc:.3f}")
    print(f"Precision (yes): {precision:.3f}")
    print(f"Recall    (yes): {recall:.3f}")
    print(f"F1-score (yes) : {f1:.3f}")

    return {
        "model": name,
        "roc_auc": roc,
        "precision_yes": precision,
        "recall_yes": recall,
        "f1_yes": f1,
        "pipeline": pipeline,
        "y_pred": y_pred,
        "y_proba": y_proba
    }


In [ ]:
# Train and evaluate all models, then build a comparison table

results = []

for name, pipe in models.items():
    metrics = evaluate_model(name, pipe, X_train, y_train, X_test, y_test)
    results.append(metrics)

results_table = pd.DataFrame(
    [
        {
            "model": r["model"],
            "roc_auc": r["roc_auc"],
            "precision_yes": r["precision_yes"],
            "recall_yes": r["recall_yes"],
            "f1_yes": r["f1_yes"],
        }
        for r in results
    ]
).sort_values(by="roc_auc", ascending=False)

print("\n\n=== Model comparison (sorted by ROC-AUC) ===")
display(results_table)


In [ ]:
# ROC curve comparison for all models

plt.figure(figsize=(7, 7))

for r in results:
    fpr, tpr, _ = roc_curve(y_test, r["y_proba"])
    plt.plot(fpr, tpr, label=f'{r["model"]} (AUC={r["roc_auc"]:.3f})')

plt.plot([0, 1], [0, 1], 'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - All Models")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
# Confusion matrix for the best model (by ROC-AUC)

best = max(results, key=lambda r: r["roc_auc"])
print(f"\nBest model according to ROC-AUC: {best['model']}")

cm = confusion_matrix(y_test, best["y_pred"])

plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {best['model']}")
plt.tight_layout()
plt.show()


In [ ]:
# cross-validated ROC-AUC on the training set for all models

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nCross-validated ROC-AUC on training set:")

for name, pipe in models.items():
    cv_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )
    print(f"{name}: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")



## 7. Feature Importance & Interpretation

In this section, we:
- Extract feature names from the preprocessing pipeline
- Compute feature importance for a tree-based model (e.g., Random Forest or XGBoost)
- Visualize the top features
- Interpret the main drivers of subscription


In [ ]:
# Fit the preprocessor alone on the training data to retrieve feature names

preprocessor.fit(X_train)

# Numeric feature names (unchanged by StandardScaler)
num_features_out = numeric_features

# Categorical feature names after one-hot encoding
cat_transformer = preprocessor.named_transformers_['cat']
cat_feature_names = cat_transformer.get_feature_names_out(categorical_features)

all_feature_names = list(num_features_out) + list(cat_feature_names)
len(all_feature_names), all_feature_names[:10]


In [ ]:
# Choose a tree-based model for feature importance
# Here, we use the Random Forest model, but you could also use XGBoost.

rf_pipeline = models["Random Forest"]
rf_model = rf_pipeline.named_steps['model']

importances = rf_model.feature_importances_

feat_importances = pd.DataFrame({
    'feature': all_feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

feat_importances.head(20)


In [ ]:
# Plot top 20 important features

top_n = 20

plt.figure(figsize=(8, 10))
sns.barplot(
    data=feat_importances.head(top_n),
    x='importance',
    y='feature'
)
plt.title(f"Top {top_n} Feature Importances - Random Forest")
plt.tight_layout()
plt.show()



## Business Interpretation
From the empirical feature importance analysis and the performance of all evaluated models, several meaningful business patterns emerge:

### **1. Macroeconomic indicators are among the strongest predictors**

The top-ranked features include:

* **euribor3m**
* **nr.employed**
* **emp.var.rate**
* **cons.conf.idx**
* **cons.price.idx**

These variables capture economic stability, consumer sentiment, and interest rate environment.
**When interest rates are favorable or economic confidence is low, customers are more likely to prefer the safety of term deposits.**
This confirms that subscription rates are **highly dependent on market conditions**, and campaign timing must account for macroeconomic signals.


### **2. Age is the single strongest demographic predictor**

The feature importance ranking places **age** at the top.

Older customers—especially those above 50—tend to:

* have more stable income,
* prefer capital protection,
* seek predictable returns.

This suggests a strong case for **age-based segmentation** and differentiated messaging.


### **3. Campaign-related variables show clear diminishing returns**

Variables like:

* **campaign** (current number of contacts)
* **previous** (historic number of contacts)
* **pdays** (days since last contact)

indicate that:

* The more times a customer is contacted,
* **the less likely they are to subscribe**.

This signals **customer fatigue** and diminishing marginal effectiveness.
Repeated calls should be limited, especially for low-propensity customers.


### **4. Outcome of previous campaign is a strong driver**

The category **poutcome_success** is among the most important features.

Customers who converted in a previous campaign show **much higher probability** of subscribing again.

This creates a valuable “warm lead” segment.


### **5. Housing loan and personal loan status contain useful signals**

Features such as **housing_yes / housing_no** appear in the top 20.

This may reflect:

* liquidity constraints,
* debt burden,
* financial stability.

Customers without major loans may be **more open to term deposits**, given higher disposable income.


### **6. Day of week effects appear consistently**

Surprisingly, variables like **day_of_week_mon, fri, wed, thu** have non-negligible importance.

This suggests:

* customer responsiveness varies with weekday patterns
* Mondays, Thursdays, and Fridays show meaningful predictive value

This can be tested operationally with **temporal optimization of campaigns**.


### **Overall Interpretation**

These findings reveal that subscription propensity is driven by a combination of:

* **Macroeconomic environment (very strong)**
* **Customer demographics (age, job, education)**
* **Behavioral history and previous outcomes**
* **Contact strategy effectiveness**
* **Weekly timing patterns**

These insights provide clear guidance for **customer segmentation, campaign timing, and targeted communication strategies**.



## **7. Conclusions & Next Steps**

### **Conclusions**

* The dataset is strongly imbalanced (~11% positive class).
* The variable **duration** remains the strongest predictor overall but cannot be used in live predictions due to **target leakage**.
* In production-ready models, the strongest features were:

  * **Macroeconomic indicators** (euribor3m, nr.employed, emp.var.rate)
  * **Age**
  * **Campaign/contact frequency**
  * **Previous campaign success**
  * **Education, marital status, job categories**
* Among the evaluated models:

  * **Gradient Boosting achieved the highest ROC-AUC (0.810)** but had low recall.
  * **XGBoost achieved similar ROC-AUC (0.809)** and the **highest recall (0.652)** and **best F1-score (0.485)** for the positive class.
  * **Logistic Regression performed competitively**, showing strong recall compared to tree models.
  * **Random Forest had the lowest recall** and is less suitable under this business objective.

Given that the business goal emphasizes **identifying likely subscribers before calling**,
**recall on the positive class is critically important** (failing to call a potential subscriber is more costly than calling a non-subscriber).

Therefore:

###  **Best overall model for business impact: XGBoost**

Because it:

* preserves competitive ROC-AUC,
* achieves the highest recall among all models,
* produces the best F1 score,
* handles class imbalance effectively with scale_pos_weight,
* and offers feature importance interpretability.





## **Next Steps**

### **1. Deploy XGBoost as a scoring service (Azure or AWS)**

* Export model as a serialized artifact (`.json` or `.pkl`).
* Host as REST API behind Azure ML or AWS SageMaker.

### **2. Integrate with CRM and dialing systems**

Use model probabilities to rank clients:

* Tier 1: High-propensity
* Tier 2: Medium-propensity
* Tier 3: Low-propensity (limit call attempts)

### **3. Implement A/B testing**

Compare:

* Traditional calling strategy
  vs.
* Model-driven prioritization

Measure uplift in:

* conversion rate
* cost per acquisition
* average handling time

### **4. Monitor and retrain periodically**

Due to macroeconomic importance, retrain:

* every 3 months
  or
* when interest rates/consumer confidence shift significantly.

### **5. Expand segmentation strategy using insights**

Leverage age, job, education, and previous success to design:

* personalized scripts
* priority segments
* optimal calling days



# Additional Visualizations

### Class Imbalance Visualization

In [ ]:
### Class Imbalance Visualization

plt.figure(figsize=(6,4))
sns.countplot(x=df['y'], palette='Set2')
plt.title("Target Distribution (y)")
plt.xlabel("Subscription")
plt.ylabel("Count")
plt.show()

print(df['y'].value_counts(normalize=True))


### Distribution of Numeric Features by Subscription Outcome

In [ ]:
 ### Distribution of Numeric Features by Target

numeric_cols_plot = ['age','campaign','pdays','previous',
                     'emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m']

fig, axes = plt.subplots(4, 2, figsize=(12, 16))
axes = axes.flatten()

for i, col in enumerate(numeric_cols_plot):
    sns.kdeplot(data=df, x=col, hue='y', fill=True, ax=axes[i], palette="Set2")
    axes[i].set_title(f"Distribution of {col} by subscription (y)")

plt.tight_layout()
plt.show()


 ### Extended Correlation Heatmap (Numeric Features)

In [ ]:
 ### Extended Correlation Heatmap (Numeric Features)

 # Identify all numeric columns again
numeric_cols_all = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create binary target for correlation analysis
df['y_binary'] = (df['y'] == 'yes').astype(int)


plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols_all + ['y_binary']].corr(),
            annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap: Numeric Features + Target")
plt.show()


### Subscription Rate by Job Category

In [ ]:
### Subscription Rate by Job

sr_job = df.groupby('job')['y'].apply(lambda x: (x=='yes').mean()).sort_values()

plt.figure(figsize=(10,6))
sr_job.plot(kind='barh', color='steelblue')
plt.title("Subscription Rate by Job Category")
plt.xlabel("Rate of 'yes'")
plt.show()

sr_job


### Subscription Rate by Education

In [ ]:
### Subscription Rate by Education

sr_edu = df.groupby('education')['y'].apply(lambda x: (x=='yes').mean()).sort_values()

plt.figure(figsize=(10,6))
sr_edu.plot(kind='barh', color='forestgreen')
plt.title("Subscription Rate by Education Level")
plt.xlabel("Rate of 'yes'")
plt.show()

sr_edu


### Subscription by Age Group

In [ ]:
### Subscription by Age Group

df_age = df.copy()
df_age['age_group'] = pd.cut(df_age['age'],
                             bins=[17, 25, 35, 50, 65, 100],
                             labels=["18–25", "26–35", "36–50", "51–65", "65+"])

sr_age = df_age.groupby('age_group')['y'].apply(lambda x: (x=='yes').mean())

plt.figure(figsize=(7,4))
sr_age.plot(kind='bar', color='darkcyan')
plt.title("Subscription Rate by Age Group")
plt.ylabel("Rate of 'yes'")
plt.show()

sr_age


In [ ]:
### PCA Visualization of Encoded Features

from sklearn.decomposition import PCA

# Fit preprocessor on full training data
X_train_transformed = preprocessor.fit_transform(X_train)

# Reduce to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_transformed.toarray() if hasattr(X_train_transformed, "toarray") else X_train_transformed)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=y_train, cmap='coolwarm', alpha=0.3)
plt.title("PCA Projection of Encoded Features")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Subscription (0=No, 1=Yes)")
plt.show()


### Feature Importance Comparison Across Models

In [ ]:
### Feature Importance Comparison: Gradient Boosting vs. XGBoost

gb = models["Gradient Boosting"].named_steps['model']
xgb = models["XGBoost"].named_steps['model']

feat_imp_gb = pd.DataFrame({
    "feature": all_feature_names,
    "importance_gb": gb.feature_importances_
}).sort_values("importance_gb", ascending=False)

feat_imp_xgb = pd.DataFrame({
    "feature": all_feature_names,
    "importance_xgb": xgb.feature_importances_
}).sort_values("importance_xgb", ascending=False)

# Merge
feat_compare = feat_imp_gb.merge(feat_imp_xgb, on="feature")

feat_compare.head(20)


### Precision-Recall Curves for All Models

In [ ]:
### Precision-Recall Curves for All Models

from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(7,6))

for r in results:
    precision, recall, _ = precision_recall_curve(y_test, r["y_proba"])
    plt.plot(recall, precision, label=f'{r["model"]} (AUC={r["roc_auc"]:.3f})')

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves – All Models")
plt.legend()
plt.grid(True)
plt.show()


### SHAP Explainability

To ensure the model is interpretable and suitable for business decision-making, we apply **SHAP (SHapley Additive exPlanations)** to the best performing tree-based model.  
Although tree-based ensembles such as Gradient Boosting or XGBoost offer strong predictive performance, they can behave as “black boxes” without proper explainability.  
SHAP provides a principled and consistent way to interpret these models by:

- Quantifying how each feature contributes to increasing or decreasing the prediction for a specific client (local interpretability).
- Ranking features globally based on their true contribution to model output (global interpretability).
- Visualizing interactions, nonlinearities, and heterogeneity across customer segments.
- Supporting regulatory and business transparency requirements, which are critical in financial institutions.

Using SHAP allows us to go beyond feature importance and deliver **actionable, trustworthy explanations** for why the model predicts that certain clients are more or less likely to subscribe to a term deposit.  
This strengthens both the model’s usability and its acceptance by stakeholders such as marketing teams, risk teams, and compliance.


In [ ]:

import shap

# Initialize JS viz if you're in a notebook environment
shap.initjs()


In [ ]:
# We use the trained XGBoost model and the same preprocessing pipeline.
# We transform the data with `preprocessor` and then explain the XGBoost model
# in the transformed feature space.

# Extract trained pipeline and underlying XGBoost model
xgb_pipeline = models["XGBoost"]
xgb_model = xgb_pipeline.named_steps['model']

# Ensure the preprocessor is fitted (it should already be after training)
# but we can safely call fit on X_train here
preprocessor.fit(X_train)

# Transform train and test for SHAP
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

# If the output is sparse, convert to dense for SHAP
if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()
if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

X_train_transformed.shape, X_test_transformed.shape


In [ ]:
### Build SHAP TreeExplainer for XGBoost

# For performance, we can sample a subset of the training data as background
# (especially helpful if the dataset is large)
background_size = 200  # you can adjust this
if X_train_transformed.shape[0] > background_size:
    background_idx = np.random.choice(X_train_transformed.shape[0], background_size, replace=False)
    background_data = X_train_transformed[background_idx]
else:
    background_data = X_train_transformed

explainer = shap.TreeExplainer(xgb_model, data=background_data)

# Compute SHAP values for the test set
# For binary classification, shap_values can be:
# - a matrix (n_samples, n_features), or
# - a list [shap_values_for_class0, shap_values_for_class1]
raw_shap_values = explainer.shap_values(X_test_transformed)

# Handle both cases
if isinstance(raw_shap_values, list):
    # We take SHAP values for the positive class (1)
    shap_values = raw_shap_values[1]
else:
    shap_values = raw_shap_values

shap_values.shape


In [ ]:
### Global Explainability: SHAP Summary Plot (Beeswarm)

# all_feature_names must match the transformed data order (num + OHE cat)
# We already created `all_feature_names` previously:
# all_feature_names = list(num_features_out) + list(cat_feature_names)

shap.summary_plot(shap_values, X_test_transformed, feature_names=all_feature_names)


In [ ]:
### Global Explainability: SHAP Summary Plot (Beeswarm)

# all_feature_names must match the transformed data order (num + OHE cat)
# We already created `all_feature_names` previously:
# all_feature_names = list(num_features_out) + list(cat_feature_names)

shap.summary_plot(shap_values, X_test_transformed, feature_names=all_feature_names)


In [ ]:
### Local Explainability: Single Prediction (Waterfall Plot)

# Choose a specific example from the test set
idx = 0  # you can change this index

x_sample = X_test_transformed[idx]
shap_sample = shap_values[idx]

# Build an Explanation object for SHAP's waterfall plot
expl = shap.Explanation(
    values=shap_sample,
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1],
    data=x_sample,
    feature_names=all_feature_names
)

shap.plots.waterfall(expl, max_display=15)


In [ ]:
### Comparing SHAP Explanations for Two Different Clients

idx_1 = 0
idx_2 = 10  # pick another example

for i in [idx_1, idx_2]:
    x_sample = X_test_transformed[i]
    shap_sample = shap_values[i]

    expl = shap.Explanation(
        values=shap_sample,
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1],
        data=x_sample,
        feature_names=all_feature_names
    )
    print(f"\n=== SHAP Explanation for test sample index {i} ===")
    shap.plots.waterfall(expl, max_display=10)
